# Federated Generative Semantic Spaces (FLGSS)
### Scalable, Robust, and Trustworthy Federated Intelligence for IoT Networks
**Hardware Target:** NVIDIA H100 / A100 / L4 Tensor Core GPU (Google Colab High-RAM)

This notebook provides the complete, end-to-end reproducible pipeline for the framework:
> **"Federated Generative Semantic Spaces (FLGSS) v2.0"**

---

## 1. Environment & Hardware Verification (H100 GPU)

In [ ]:
# Verify GPU hardware, compute capability and VRAM
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Device Total Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    # Enable H100 Tensor Core TF32 Speedups
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("H100 TF32 & cuDNN optimizations enabled.")

## 2. Install Dependencies

In [ ]:
# Install requirements
!pip install -q -r requirements.txt
import flgss
print(f"FLGSS Package v{flgss.__version__} loaded successfully.")

--- 
## 3. Quick-Run Demo: FLGSS on CIFAR-10 (High Skew $\alpha=0.1$)

In [ ]:
# Run 10 rounds of FLGSS on CIFAR-10 with high non-IID heterogeneity
!python train.py --dataset cifar10 --algo flgss --alpha 0.1 --rounds 10 --clients 100 --C 0.1

--- 
## 4. Benchmark Suite: Reproduce Table I (Non-IID Accuracy Comparison)
Compares **FLGSS** against standard and state-of-the-art baselines:
- **FedAvg** (McMahan et al. 2017)
- **FedProx** (Li et al. 2020)
- **SCAFFOLD** (Karimireddy et al. 2020)
- **MOON** (Li et al. 2021)
- **FedSAM** (Qu et al. 2023)
- **FedKD / PFedKD** (Wu et al. 2022, Li et al. 2025)

*All baselines utilize the exact same frozen ResNet-18 backbone for architectural fairness (Fair Backbone Protocol).*

In [ ]:
# Run Table I Benchmark for CIFAR-10 and UCI-HAR
!python run_experiments.py --suite table1 --fast --rounds 20

--- 
## 5. Trustworthiness & Byzantine Robustness Evaluation (Byzantine Robustness)
Simulates a **20% Byzantine Model Poisoning Attack**:
1. Without Robust Aggregation (Filter disabled) $\rightarrow$ model collapse
2. With FLGSS Mahalanobis-based Anomaly Filter $\rightarrow$ malicious updates rejected, accuracy preserved

In [ ]:
# Run Table IV Ablation Study (Under Attack with vs. without Mahalanobis filter)
!python run_experiments.py --suite table4 --rounds 20

--- 
## 6. Unsupervised Anomaly Detection (Anomaly Detection)
Evaluates the ability of the Global Generative Model to detect out-of-distribution / novel anomalies vs. Federated Autoencoder (FedAE).

In [ ]:
!python run_experiments.py --suite table2 --rounds 15

--- 
## 7. Generate and Display Publication-Ready Figures
Generates vector PDFs and high-res PNGs generated from benchmark runs:
- `Figure_1.pdf`: Convergence curves
- `comm_cost.pdf`: Communication overhead vs. target accuracy (>11x reduction)
- `abl1.pdf`: Anchor model sensitivity
- `abl2.pdf`: Latent dimension trade-off
- `abl3.pdf`: Client-level DP privacy-utility trade-off

In [ ]:
!python run_experiments.py --suite plots

from IPython.display import Image, display, Markdown
import os

figures = [
    (
        'plots/Figure_1.png',
        '### Figure 4: CIFAR-10 Convergence Curves (200 Communication Rounds)\n'
        '**Caption:** *Test accuracy vs. communication rounds on CIFAR-10 under a highly non-IID data distribution (α = 0.1). '
        'FLGSS demonstrates significantly faster convergence and achieves a much higher final accuracy compared to baselines.*'
    ),
    (
        'plots/comm_cost.png',
        '### Figure 5: Communication Cost Comparison (Log Scale)\n'
        '**Caption:** *Total communication cost (MB, log scale) required to achieve 70% test accuracy on CIFAR-10 under non-IID skew (α = 0.1). '
        'FLGSS achieves over 11× communication reduction compared to baseline federated algorithms.*'
    ),
    (
        'plots/abl1.png',
        '### Ablation 1: Anchor Model Sensitivity\n'
        '**Caption:** *Impact of Anchor Model architecture and pretraining on final accuracy across heterogeneity levels.*'
    ),
    (
        'plots/abl2.png',
        '### Ablation 2: Latent Dimension Trade-off\n'
        '**Caption:** *Trade-off between latent representation dimension d and accuracy / computational cost.*'
    ),
    (
        'plots/abl3.png',
        '### Ablation 3: Differential Privacy Trade-off\n'
        '**Caption:** *Client-level differential privacy epsilon vs. accuracy trade-off.*'
    )
]

for img_path, caption in figures:
    if os.path.exists(img_path):
        display(Markdown(caption))
        display(Image(filename=img_path))
    else:
        print(f'File not found: {img_path}')


--- 
## 8. Export All Results and Plots

In [ ]:
# 1. Package all logs, CSVs, and generated plots into a zip file
!zip -r flgss_results.zip runs/ plots/ *.csv *.pdf *.png
print('All results packaged in flgss_results.zip.')

# 2. Automatically trigger browser download in Colab
try:
    from google.colab import files
    print('Triggering browser download...')
    files.download('flgss_results.zip')
except Exception as e:
    print(f'Note: files.download failed ({e}). Download manually from Colab left sidebar.')


In [ ]:
# Optional: Copy directly to your Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp flgss_results.zip /content/drive/MyDrive/
# print('Successfully copied flgss_results.zip to your Google Drive!')
